# **Introduction to basic Monte Carlo simulations**
### **Author: Dr. Rahul Remanan**

# Part 01 -- Measuring outcome probabilities:

If the probability of a successful outcome by trying once is: $\frac{1}{100}$, what is the probability of a successful outcome after trying $100$ times?

$Win_{probability} = 0.01$

$Loss_{probability} = 1 - Win_{probability}$

$Win_{probability} \text{ after n tries } = 1 - (Loss_{probability})^n$

$\therefore \text{When n=100, } Win_{probability} \text{ after n tries } = 1 - (Loss_{probability})^{100} = 1 - 0.99^{100}$

In [ ]:
import math, random, numpy as np

## **Evaluate outcome probabilities statistically:**

In [ ]:
p_success = 0.01
num_trials = 100

In [ ]:
p_multi_trial_success = 1 - (1 - p_success)**num_trials
print(p_multi_trial_success)

## **Evaluate outcome probabilities using Monte Carlo simulation:**

In [ ]:
num_mc_steps = 100000

mc_multi_trial_success = 0

for m in range(num_mc_steps):
    success_list = []
    for i in range(num_trials):
        success = False
        if p_success > random.SystemRandom().random():
            success = True
        success_list.append(success)
    
    success_prob = sum(success_list) / len(success_list)
    
    if m % int(num_mc_steps * 0.01) == 0 and m != 0:

        print(
            f'Success probability for trial: {m} ',
            success_prob
        )

    if sum(success_list) > 0:
        mc_multi_trial_success += 1

p_mc_multi_trial_success = mc_multi_trial_success / num_mc_steps

print(
    f'Success probability for multiple tries: ', 
    p_mc_multi_trial_success
)

In [ ]:
print(
    f'Error in computing probabilities using Monte Carlo {num_mc_steps} simulations: ', 
    p_multi_trial_success - p_mc_multi_trial_success
)

# Part 02 -- [The goat grazing problem](https://en.wikipedia.org/wiki/Goat_grazing_problem)

## **Evaluate the goat grazing problem using Monte Carlo simulations:**

For the goat to graze $\frac{1}{2}$ area of the grazing pen while tethered to a fixed point on the perimeter of the circular pen, $\frac{Radius_{\text{Goat pen}}}{Length_{\text{Goat tether}}} \approx \frac{\pi}{e}$

In [ ]:
def goat_pen(x, y, r):
    if math.pow(x, 2) + math.pow(y, 2) <= math.pow(r, 2):
        return True
    return False

In [ ]:
def goat_tether(x, y, r, xt, yt, rt):
    if goat_pen(x, y, r) and math.pow(xt, 2) + math.pow(yt, 2) <= math.pow(rt, 2):
        return True
    return False

In [ ]:
def goat_grazing_area(r=1, rt=1, num_mc_sims=1e8):
    grazing_area, pen_area = 0, 0
    for i in range(num_mc_sims):
        x  = r * random.SystemRandom().random() * random.SystemRandom().choice([1, -1])
        y  = r * random.SystemRandom().random() * random.SystemRandom().choice([1, -1])
        if goat_pen(x, y, r):
            pen_area += 1
        if goat_tether(x, y, r, r + x, y, rt):
            grazing_area += 1
    return {
        'Radius of the grazing pen': r, 
        'Length of the tether' : rt,  
        'Grazing area to the pen area ratio': grazing_area / pen_area
    }

In [ ]:
mc_rt_list = []
i, num_steps = 0, 100
approx_similarity_tolerance = 1e-2
while num_steps:
    r  = 10
    rt = r *  random.SystemRandom().uniform(math.pi / (math.exp(1) + 1e-3), math.pi / (math.exp(1) - 1e-3)) # r * math.pi / math.exp(1) #
    mc_result = goat_grazing_area(r=r, rt=rt, num_mc_sims=1e5)
    print(mc_result)
    if np.isclose(mc_result['Grazing area to the pen area ratio'], 0.5, rtol=approx_similarity_tolerance):
        i += 1
        mc_rt_list.append(mc_result['Length of the tether'])
        if len(mc_rt_list) > 0 and i >= num_steps:
            num_steps = False

In [ ]:
print(
    np.mean(mc_rt_list) / r,
    np.isclose(
        np.mean(mc_rt_list) / r,
        math.pi / math.exp(1),
        rtol=approx_similarity_tolerance
    )
)